# Model Performance Analysis

Comprehensive comparison of match prediction models across:
- 3 ML models (Logistic Regression, Random Forest, Gradient Boosting)
- 5 feature sets (ELO only → all 35 features)
- 2 pure rating baselines (ELO, Glicko-2)
- 4 metrics (Accuracy, Log Loss, Brier Score, ECE)

**Dataset:** 34,521 matches across 300 tournaments (80/20 temporal split)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.style.use('dark_background')
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'figure.facecolor': '#0a0a0a',
    'axes.facecolor': '#111113',
    'axes.edgecolor': '#333',
    'grid.color': '#222',
    'grid.alpha': 0.5,
})

GREEN = '#22c55e'
BLUE = '#3b82f6'
RED = '#ef4444'
YELLOW = '#eab308'
PURPLE = '#a855f7'
GRAY = '#71717a'

with open('../data/processed/analysis_results.json') as f:
    data = json.load(f)

results_df = pd.DataFrame(data['results'])
print(f"Loaded {len(results_df)} model configurations")
print(f"Train: {data['dataset']['train']}, Test: {data['dataset']['test']}")

## 1. Accuracy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

# Exclude coin flip for readability
plot_df = results_df[results_df['model'] != 'Coin flip'].copy()
plot_df = plot_df.sort_values('accuracy')
plot_df['label'] = plot_df['model'] + ' | ' + plot_df['features']

colors = []
for _, row in plot_df.iterrows():
    if 'Pure' in row['model']:
        colors.append(GRAY)
    elif row['model'] == 'Gradient Boosting':
        colors.append(GREEN)
    elif row['model'] == 'Logistic Regression':
        colors.append(BLUE)
    else:
        colors.append(PURPLE)

bars = ax.barh(range(len(plot_df)), plot_df['accuracy'], color=colors, height=0.7)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['label'], fontsize=9)
ax.set_xlabel('Accuracy')
ax.set_title('Match Prediction Accuracy by Model and Feature Set')
ax.axvline(x=0.5, color=RED, linestyle='--', alpha=0.5, label='Coin flip (50%)')
ax.set_xlim(0.65, 0.71)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))

# Annotate best
best = plot_df.iloc[-1]
ax.annotate(f"{best['accuracy']*100:.2f}%", xy=(best['accuracy'], len(plot_df)-1),
            xytext=(10, 0), textcoords='offset points', fontweight='bold', color=GREEN)

ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../data/processed/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Multi-Metric Comparison (Best of Each Model)

In [ ]:
# Best configuration per model type
best_per_model = results_df.loc[results_df.groupby('model')['accuracy'].idxmax()]
best_per_model = best_per_model.sort_values('accuracy', ascending=False)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metrics = [
    ('accuracy', 'Accuracy (↑)', True),
    ('log_loss', 'Log Loss (↓)', False),
    ('brier_score', 'Brier Score (↓)', False),
    ('ece', 'ECE (↓)', False),
]

model_colors = {
    'Gradient Boosting': GREEN,
    'Logistic Regression': BLUE,
    'Random Forest': PURPLE,
    'Pure ELO': YELLOW,
    'Pure Glicko-2': '#f97316',
    'Coin flip': RED,
}

for ax, (metric, title, higher_better) in zip(axes, metrics):
    df_sorted = best_per_model.sort_values(metric, ascending=not higher_better)
    colors = [model_colors.get(m, GRAY) for m in df_sorted['model']]
    ax.barh(range(len(df_sorted)), df_sorted[metric], color=colors, height=0.6)
    ax.set_yticks(range(len(df_sorted)))
    ax.set_yticklabels(df_sorted['model'], fontsize=8)
    ax.set_title(title, fontsize=11)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Best Configuration per Model — All Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../data/processed/multi_metric_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Set Impact (Gradient Boosting)

In [ ]:
gb_df = results_df[results_df['model'] == 'Gradient Boosting'].sort_values('accuracy')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by feature set
colors_fs = [GREEN if i == len(gb_df)-1 else BLUE for i in range(len(gb_df))]
ax1.barh(range(len(gb_df)), gb_df['accuracy'], color=colors_fs, height=0.6)
ax1.set_yticks(range(len(gb_df)))
ax1.set_yticklabels(gb_df['features'], fontsize=10)
ax1.set_xlabel('Accuracy')
ax1.set_title('Gradient Boosting: Accuracy by Feature Set')
ax1.set_xlim(0.68, 0.705)
ax1.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))
for i, (_, row) in enumerate(gb_df.iterrows()):
    ax1.text(row['accuracy'] + 0.0003, i, f"{row['accuracy']*100:.2f}%",
             va='center', fontsize=9, color='white')

# Log Loss by feature set
ax2.barh(range(len(gb_df)), gb_df['log_loss'], color=colors_fs, height=0.6)
ax2.set_yticks(range(len(gb_df)))
ax2.set_yticklabels(gb_df['features'], fontsize=10)
ax2.set_xlabel('Log Loss (lower = better)')
ax2.set_title('Gradient Boosting: Log Loss by Feature Set')
for i, (_, row) in enumerate(gb_df.iterrows()):
    ax2.text(row['log_loss'] + 0.001, i, f"{row['log_loss']:.4f}",
             va='center', fontsize=9, color='white')

plt.tight_layout()
plt.savefig('../data/processed/feature_set_impact.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Feature Importance (Gradient Boosting, All Features)

In [ ]:
fi = pd.DataFrame(data['feature_importance'])
fi_top = fi.head(15).sort_values('importance')

fig, ax = plt.subplots(figsize=(10, 7))
colors_fi = [GREEN if v > 0.1 else BLUE if v > 0.02 else GRAY
             for v in fi_top['importance']]
ax.barh(range(len(fi_top)), fi_top['importance'], color=colors_fi, height=0.7)
ax.set_yticks(range(len(fi_top)))
ax.set_yticklabels(fi_top['feature'], fontsize=10)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Feature Importances (Gradient Boosting)')

for i, (_, row) in enumerate(fi_top.iterrows()):
    ax.text(row['importance'] + 0.002, i, f"{row['importance']:.3f}",
            va='center', fontsize=9, color='white')

plt.tight_layout()
plt.savefig('../data/processed/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Calibration Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Perfect calibration line
ax.plot([0, 1], [0, 1], 'w--', alpha=0.4, label='Perfect calibration')

cal_colors = {
    'Gradient Boosting | Ratings combined (6)': (GREEN, 'GB (Ratings combined)'),
    'Gradient Boosting | All features (35)': (BLUE, 'GB (All features)'),
    'Pure ELO': (YELLOW, 'Pure ELO'),
    'Pure Glicko-2': ('#f97316', 'Pure Glicko-2'),
}

for key, (color, label) in cal_colors.items():
    if key in data['calibration']:
        cal = data['calibration'][key]
        # Only plot bins with enough data
        mask = [c > 50 for c in cal['counts']]
        centers = [c for c, m in zip(cal['centers'], mask) if m]
        actual = [a for a, m in zip(cal['actual'], mask) if m]
        ax.plot(centers, actual, 'o-', color=color, label=label, markersize=6, linewidth=2)

ax.set_xlabel('Predicted Probability')
ax.set_ylabel('Actual Frequency')
ax.set_title('Calibration Plot: Predicted vs Actual Win Probability')
ax.legend(loc='upper left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/calibration_plot.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. ELO vs Glicko-2 Rating Comparison

In [ ]:
from snooker_elo.data.loader import load_matches
from snooker_elo.ratings.elo import EloRating
from snooker_elo.ratings.glicko2 import Glicko2Rating

matches = load_matches()
elo = EloRating(k_factor=9.77, divisor=327.15)
elo.update(matches)
g2 = Glicko2Rating(tau=1.488, default_rating=1500)
g2.update(matches)

# Get ratings for players with 100+ matches
common = []
for name in elo.players:
    ep = elo.players[name]
    gp = g2.players.get(name)
    if ep.matches_played >= 100 and gp:
        common.append((name, ep.rating, gp.rating, gp.rd))

df_compare = pd.DataFrame(common, columns=['name', 'elo', 'glicko2', 'rd'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Scatter: ELO vs Glicko-2
sc = ax1.scatter(df_compare['elo'], df_compare['glicko2'], 
                 c=df_compare['rd'], cmap='RdYlGn_r', s=20, alpha=0.7)
ax1.set_xlabel('ELO Rating')
ax1.set_ylabel('Glicko-2 Rating')
ax1.set_title(f'ELO vs Glicko-2 ({len(df_compare)} players, 100+ matches)')
plt.colorbar(sc, ax=ax1, label='Rating Deviation (RD)')

# Top players annotation
for _, row in df_compare.nlargest(5, 'elo').iterrows():
    ax1.annotate(row['name'].split()[-1], (row['elo'], row['glicko2']),
                 fontsize=8, color='white', alpha=0.8,
                 xytext=(5, 5), textcoords='offset points')

# RD distribution
ax2.hist(df_compare['rd'], bins=30, color=BLUE, alpha=0.8, edgecolor='#333')
ax2.axvline(df_compare['rd'].median(), color=GREEN, linestyle='--', label=f"Median: {df_compare['rd'].median():.0f}")
ax2.set_xlabel('Rating Deviation (RD)')
ax2.set_ylabel('Number of Players')
ax2.set_title('Glicko-2 RD Distribution')
ax2.legend()

plt.tight_layout()
plt.savefig('../data/processed/elo_vs_glicko2.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation
print(f"ELO-Glicko2 correlation: {df_compare['elo'].corr(df_compare['glicko2']):.4f}")

## 7. Summary

### Key Findings

| Finding | Detail |
|---------|--------|
| **Best model** | Gradient Boosting + Ratings combined (6 features) = **70.2% accuracy** |
| **Diminishing returns** | 35 features barely beats 6 features (70.0% vs 70.2%) |
| **Most important features** | `elo_match_win_rate` (45.7%) and `glicko2_match_win_rate` (28.9%) dominate |
| **Glicko-2 RD matters** | 4 Glicko-2 features (with RD) = 69.7%, nearly matching full models |
| **ELO vs Glicko-2** | ELO slightly better on accuracy, Glicko-2 better on calibration |
| **Best calibration** | GB + Ratings combined: ECE = 0.011 (nearly perfect) |
| **Ceiling** | ~70% appears to be the practical ceiling for match prediction from ratings alone |

### Why 70% is the Ceiling

Professional snooker has inherent randomness — even the best player only wins ~56% of frames against a top opponent. Day-to-day form, mental state, and match conditions create noise that no rating system can capture. The 70% accuracy is comparable to bookmaker accuracy for tennis and other individual sports.